# 03 — Sequential historical extraction

F0 only. Protocol v1.5 + implementation lock v1.5k.


In [2]:
from pathlib import Path
import sys, pandas as pd
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
from ad_f0.io import load_config, ensure_dirs
cfg=load_config(ROOT); ensure_dirs(ROOT,cfg)
print('ROOT =',ROOT)


ROOT = C:\Users\uysal\AD_ROUTER_CP_F0_v1.8


In [3]:
# After preflight only. If True, missing required SQLite archives are downloaded one at a time.
AUTO_DOWNLOAD_MISSING = True
RETAIN_BULK_ARCHIVES = False
KEEP_EXTRACTED_DB_FILES = True

# Operational only: valid release-level checkpoints may be reused after a kernel/browser interruption.
RESUME_VALID_CHECKPOINTS = True
EXTRACTION_POLICY_VERSION = "F0-v1.5k-pre15-native-assay-ambiguity-confidence-regime-r1"

required=pd.read_csv(ROOT/'results'/'audit'/'chembl_release_manifest_required.csv',dtype={'release':str})
required['exact_release_date']=pd.to_datetime(required['exact_release_date'])
bridge=pd.read_parquet(ROOT/'data'/'interim'/'f0_pre9_target_sequence_bridge.parquet')
print('Required historical snapshots:',len(required))


Required historical snapshots: 33


In [4]:
import gc
import json
import time
from pandas.errors import EmptyDataError

from ad_f0.archive_download import download_extract_sqlite
from ad_f0.chembl_adapter import (
    extract_release, schema_probe_row, audit_pre9_target_bridge, MultiComponentThresholdError
)
from ad_f0.audits import unit_conversion_audit
from ad_f0.io import sha256_file
from ad_f0.standardize import configure_rdkit_logging, clear_standardization_caches

# Suppress per-molecule INFO chatter; warnings/errors (including tautomer-limit warnings) remain visible.
configure_rdkit_logging(show_info=False)

base_manifest=pd.read_csv(ROOT/cfg['paths']['release_manifest'],dtype={'release':str})
local_lookup={str(r.release_label):str(r.local_db_path).strip() for _,r in base_manifest.iterrows()
              if pd.notna(r.local_db_path) and str(r.local_db_path).strip()}
probe_download=ROOT/'results'/'audit'/'f0_probe_download_manifest.csv'
if probe_download.exists():
    for _,r in pd.read_csv(probe_download,dtype={'release':str}).iterrows():
        if pd.notna(r.get('db_path')): local_lookup[str(r.release_label)]=str(r.db_path)

extract_dir=ROOT/cfg['paths']['extracted_dir']; extract_dir.mkdir(parents=True,exist_ok=True)
checkpoint_root=ROOT/'results'/'checkpoints'/'f0_extract'; checkpoint_root.mkdir(parents=True,exist_ok=True)


def _safe_read_csv(path):
    try:
        return pd.read_csv(path)
    except (FileNotFoundError, EmptyDataError):
        return pd.DataFrame()


def _write_sidecar(df, path):
    path.parent.mkdir(parents=True,exist_ok=True)
    df.to_csv(path,index=False)


def _sidecar(label, kind):
    return checkpoint_root/label/f'{kind}.csv'


def _checkpoint_is_valid(label, db_hash, parquet_path):
    meta_path=checkpoint_root/label/'checkpoint.json'
    if not (RESUME_VALID_CHECKPOINTS and meta_path.exists() and parquet_path.exists()):
        return False
    try:
        meta=json.loads(meta_path.read_text(encoding='utf-8'))
        return (
            meta.get('policy_version')==EXTRACTION_POLICY_VERSION
            and meta.get('db_sha256')==db_hash
            and meta.get('parquet_sha256')==sha256_file(parquet_path)
        )
    except Exception:
        return False


for _,r in required.sort_values('exact_release_date').iterrows():
    label=str(r.release_label); version=str(r.release)
    raw=local_lookup.get(label,'')
    db=Path(raw) if raw else None
    if db is not None and not db.is_absolute(): db=ROOT/db
    downloaded=False; got=None
    if db is None or not db.exists():
        if not AUTO_DOWNLOAD_MISSING:
            raise RuntimeError(f'{label} DB missing. Set local_db_path or set AUTO_DOWNLOAD_MISSING=True after reviewing the bulk plan.')
        got=download_extract_sqlite(version,ROOT/cfg['paths']['archives_dir']/'bulk_archives',ROOT/cfg['paths']['cache_dir']/'bulk_sqlite',retain_archive=RETAIN_BULK_ARCHIVES)
        db=got.db_path; downloaded=True

    db_hash=sha256_file(db)
    parquet_path=extract_dir/f'{label}_activities.parquet'
    if _checkpoint_is_valid(label,db_hash,parquet_path):
        print(f'Checkpoint valid; skipping {label}: {parquet_path}')
        continue

    print('Extracting',label,db)
    started=time.time()
    probe=schema_probe_row(db,label)
    major=float(version.replace('_','.'))
    expected='pre9_early' if major<9 else 'pre15_legacy' if major<15 else 'post15'
    if probe['schema_generation']!=expected or not bool(probe['schema_structural_probe_pass']):
        raise RuntimeError(f'{label}: schema mismatch/structural failure: {probe}')
    if not bool(probe.get('target_type_semantics_probe_pass',False)):
        raise RuntimeError(
            f"{label}: historical target-type equivalence failed: requested={probe.get('requested_scientific_target_type')} "
            f"native={probe.get('historical_native_target_type_equivalent')} "
            f"human_target_count={probe.get('target_type_equivalent_human_target_count')}"
        )
    if expected=='pre9_early' and not bool(probe['primary_semantics_probe_pass']):
        raise RuntimeError(f'{label}: early primary direct-target semantics are uncertified; no post-hoc equivalence is allowed.')

    target_bridge=bridge if expected=='pre9_early' else None
    ba=pd.DataFrame()
    if expected=='pre9_early':
        ba=audit_pre9_target_bridge(db,bridge,cfg['chembl']['organism']); ba['release_label']=label

    try:
        df,audit=extract_release(
            db,endpoints=tuple(cfg['chembl']['endpoints']),
            extraction_min_confidence=cfg['chembl']['extraction_min_confidence_score'],
            organism=cfg['chembl']['organism'],target_type=cfg['chembl']['target_type'],
            primary_confidence=cfg['chembl']['primary_confidence_score'],primary_assay_type=cfg['chembl']['primary_assay_type'],
            release_label=label,multicomponent_max_fraction=cfg['chembl']['multicomponent_exclusion_max_fraction'],
            return_audit=True,target_bridge=target_bridge,
            structure_cache_path=ROOT/cfg['paths']['structure_cache_db'],
            standardization_progress=True)
    except MultiComponentThresholdError as e:
        if not e.audit.empty: e.audit.to_csv(ROOT/'results'/'audit'/f'{label}_multicomponent_threshold_failure.csv',index=False)
        raise

    # Write the parquet transactionally; an interrupted temp file can never become a valid checkpoint.
    tmp_path=parquet_path.with_name(parquet_path.name+'.tmp')
    df.to_parquet(tmp_path,index=False)
    tmp_path.replace(parquet_path)

    multi_df=audit['multicomponent_exclusions'] if not audit['multicomponent_exclusions'].empty else pd.DataFrame()
    pre15_multi_df=audit.get('pre15_multitarget_assay_exclusions',pd.DataFrame())
    mapping_df=audit['legacy_target_mapping'] if not audit['legacy_target_mapping'].empty else pd.DataFrame()
    std_df=audit.get('standardization_stats',pd.DataFrame())
    unit_df=unit_conversion_audit(df,label)
    quality_df=pd.DataFrame([{
        'release_label':label,'schema_generation':expected,
        'potential_duplicate_field_available':bool(probe['potential_duplicate_available']),
        'data_validity_comment_field_available':bool(probe['data_validity_comment_available']),
        'extracted_rows':len(df),
        'requested_scientific_target_type':probe.get('requested_scientific_target_type'),
        'historical_native_target_type_equivalent':probe.get('historical_native_target_type_equivalent'),
        'target_type_equivalent_human_target_count':probe.get('target_type_equivalent_human_target_count'),
    }])

    legacy=df[df.assay_identity_key.fillna('').astype(str).str.startswith('LEGACY_ASSAY_FP:')] if 'assay_identity_key' in df else pd.DataFrame()
    assay_df=pd.DataFrame()
    if not legacy.empty:
        assay_df=(legacy.groupby('assay_identity_key',dropna=False).agg(
            rows=('assay_identity_key','size'),
            native_assay_ids=('native_assay_id',lambda x: x.dropna().nunique()),
            documents=('document_identity_key','nunique')).reset_index())
        assay_df['release_label']=label; assay_df['native_id_collision']=assay_df.native_assay_ids.gt(1)

    summary_df=pd.DataFrame([{
        'release_label':label,'schema_generation':expected,'rows':len(df),
        'primary_record_eligible':int(df.primary_record_eligible.sum()),
        'relaxed_confidence_B_record_eligible':int(df.relaxed_confidence_B_record_eligible.sum()),
        'broad_prior_knowledge_eligible':int(df.broad_prior_knowledge_eligible.sum()),
        'pre15_multitarget_assays_excluded':int(pre15_multi_df['excluded_assays'].iloc[0]) if not pre15_multi_df.empty else 0,
        'post15_multicomponent_assays_excluded':int(multi_df['excluded_assays'].iloc[0]) if not multi_df.empty else 0,
        'unit_convertible':int(df.unit_convertible.sum()),
        'duplicate_flagged':int(df.potential_duplicate_flagged.sum()),
        'requested_scientific_target_type':'SINGLE PROTEIN',
        'historical_native_target_type_equivalent':probe.get('historical_native_target_type_equivalent'),
    }])
    source_df=pd.DataFrame([{
        'release_label':label,'release':version,'db_path':str(db),'db_sha256':db_hash,
        'downloaded_this_run':downloaded,'download_url':got.url if got else 'preconfigured_or_probe'
    }])
    schema_df=pd.DataFrame([probe])

    # Persist every audit immediately. checkpoint.json is written LAST and is the commit marker.
    release_dir=checkpoint_root/label; release_dir.mkdir(parents=True,exist_ok=True)
    _write_sidecar(summary_df,_sidecar(label,'summary'))
    _write_sidecar(schema_df,_sidecar(label,'schema'))
    _write_sidecar(quality_df,_sidecar(label,'quality'))
    _write_sidecar(source_df,_sidecar(label,'source'))
    _write_sidecar(unit_df,_sidecar(label,'unit'))
    _write_sidecar(multi_df,_sidecar(label,'multi'))
    _write_sidecar(pre15_multi_df,_sidecar(label,'pre15_multi'))
    _write_sidecar(mapping_df,_sidecar(label,'mapping'))
    _write_sidecar(std_df,_sidecar(label,'standardization'))
    _write_sidecar(ba,_sidecar(label,'bridge'))
    _write_sidecar(assay_df,_sidecar(label,'assay'))

    parquet_hash=sha256_file(parquet_path)
    meta={
        'policy_version':EXTRACTION_POLICY_VERSION,
        'release_label':label,
        'db_sha256':db_hash,
        'parquet_sha256':parquet_hash,
        'rows':len(df),
        'primary_record_eligible':int(df.primary_record_eligible.sum()),
        'schema_generation':expected,
        'historical_native_target_type_equivalent':probe.get('historical_native_target_type_equivalent'),
    }
    (release_dir/'checkpoint.json').write_text(json.dumps(meta,indent=2,sort_keys=True),encoding='utf-8')

    clear_standardization_caches()
    gc.collect()
    print(f"Completed {label}: rows={len(df):,}, primary={int(df.primary_record_eligible.sum()):,}, elapsed={time.time()-started:.1f}s")

    if downloaded and not KEEP_EXTRACTED_DB_FILES and label not in cfg['chembl']['schema_probe_releases']:
        db.unlink(missing_ok=True)


# Rebuild consolidated outputs entirely from committed release sidecars, so a resumed run is identical
# to an uninterrupted run.
def _collect(kind):
    frames=[]
    for label in required.sort_values('exact_release_date').release_label.astype(str):
        z=_safe_read_csv(_sidecar(label,kind))
        if not z.empty: frames.append(z)
    return pd.concat(frames,ignore_index=True) if frames else pd.DataFrame()

summary_df=_collect('summary')
if len(summary_df)!=len(required):
    raise RuntimeError(f'Extraction incomplete: committed summaries={len(summary_df)} required={len(required)}')

summary_df.to_csv(ROOT/'results'/'audit'/'release_extraction_summary.csv',index=False)
_collect('schema').to_csv(ROOT/'results'/'audit'/'f0_all_release_schema_audit.csv',index=False)
_collect('quality').to_csv(ROOT/'results'/'tables'/'f0_historical_quality_field_availability.csv',index=False)
_collect('source').to_csv(ROOT/'results'/'audit'/'f0_bulk_source_hashes.csv',index=False)
_collect('unit').to_csv(ROOT/'results'/'tables'/'f0_unit_conversion_audit.csv',index=False)
_collect('multi').to_csv(ROOT/'results'/'tables'/'f0_multicomponent_exclusions.csv',index=False)
_collect('pre15_multi').to_csv(ROOT/'results'/'tables'/'f0_pre15_multitarget_assay_exclusions.csv',index=False)
_collect('mapping').to_csv(ROOT/'results'/'tables'/'f0_legacy_target_mapping_audit.csv',index=False)
_collect('standardization').to_csv(ROOT/'results'/'audit'/'f0_structure_standardization_runtime.csv',index=False)
_collect('bridge').to_csv(ROOT/'results'/'tables'/'f0_pre9_target_bridge_audit_all_releases.csv',index=False)
_collect('assay').to_csv(ROOT/'results'/'tables'/'f0_legacy_assay_identity_audit.csv',index=False)

display(summary_df)
print('Historical extraction complete. Next: 04_F0_first_availability.ipynb')


[10:00:32] Initializing Normalizer


Checkpoint valid; skipping CHEMBL01: C:\Users\uysal\AD_ROUTER_CP_F0_v1.8\data\extracted\CHEMBL01_activities.parquet
Checkpoint valid; skipping CHEMBL02: C:\Users\uysal\AD_ROUTER_CP_F0_v1.8\data\extracted\CHEMBL02_activities.parquet
Checkpoint valid; skipping CHEMBL03: C:\Users\uysal\AD_ROUTER_CP_F0_v1.8\data\extracted\CHEMBL03_activities.parquet
Checkpoint valid; skipping CHEMBL04: C:\Users\uysal\AD_ROUTER_CP_F0_v1.8\data\extracted\CHEMBL04_activities.parquet
Checkpoint valid; skipping CHEMBL05: C:\Users\uysal\AD_ROUTER_CP_F0_v1.8\data\extracted\CHEMBL05_activities.parquet
Checkpoint valid; skipping CHEMBL06: C:\Users\uysal\AD_ROUTER_CP_F0_v1.8\data\extracted\CHEMBL06_activities.parquet
Checkpoint valid; skipping CHEMBL07: C:\Users\uysal\AD_ROUTER_CP_F0_v1.8\data\extracted\CHEMBL07_activities.parquet
Checkpoint valid; skipping CHEMBL08: C:\Users\uysal\AD_ROUTER_CP_F0_v1.8\data\extracted\CHEMBL08_activities.parquet
Checkpoint valid; skipping CHEMBL09: C:\Users\uysal\AD_ROUTER_CP_F0_v1.8

[10:44:50] Tautomer enumeration stopped at 1000 tautomers: max tautomers reached


CHEMBL25: standardized 250/16163 new unique smiles inputs (488884 persistent-cache hits; 32.7s)


[10:45:20] Tautomer enumeration stopped at 297 tautomers: max transforms reached
[10:45:22] Tautomer enumeration stopped at 279 tautomers: max transforms reached
[10:45:24] Tautomer enumeration stopped at 279 tautomers: max transforms reached
[10:45:26] Tautomer enumeration stopped at 279 tautomers: max transforms reached
[10:45:27] Tautomer enumeration stopped at 279 tautomers: max transforms reached
[10:45:29] Tautomer enumeration stopped at 279 tautomers: max transforms reached
[10:45:32] Tautomer enumeration stopped at 297 tautomers: max transforms reached
[10:45:35] Tautomer enumeration stopped at 279 tautomers: max transforms reached
[10:45:37] Tautomer enumeration stopped at 279 tautomers: max transforms reached
[10:45:38] Tautomer enumeration stopped at 297 tautomers: max transforms reached
[10:45:39] Tautomer enumeration stopped at 279 tautomers: max transforms reached
[10:45:42] Tautomer enumeration stopped at 279 tautomers: max transforms reached
[10:45:43] Tautomer enumerat

...

CHEMBL31: standardized 10250/10750 new unique smiles inputs (593033 persistent-cache hits; 612.2s)
CHEMBL31: standardized 10500/10750 new unique smiles inputs (593033 persistent-cache hits; 615.7s)


[11:21:10] Tautomer enumeration stopped at 1000 tautomers: max tautomers reached
[11:21:11] Tautomer enumeration stopped at 1000 tautomers: max tautomers reached
[11:21:12] Tautomer enumeration stopped at 1000 tautomers: max tautomers reached
[11:21:13] Tautomer enumeration stopped at 1000 tautomers: max tautomers reached
[11:21:14] Tautomer enumeration stopped at 1000 tautomers: max tautomers reached
[11:21:14] Tautomer enumeration stopped at 1000 tautomers: max tautomers reached
[11:21:15] Tautomer enumeration stopped at 1000 tautomers: max tautomers reached
[11:21:16] Tautomer enumeration stopped at 1000 tautomers: max tautomers reached
[11:21:17] Tautomer enumeration stopped at 1000 tautomers: max tautomers reached
[11:21:18] Tautomer enumeration stopped at 1000 tautomers: max tautomers reached
[11:21:19] Tautomer enumeration stopped at 1000 tautomers: max tautomers reached
[11:21:20] Tautomer enumeration stopped at 1000 tautomers: max tautomers reached
[11:21:21] Tautomer enumerat

CHEMBL31: standardized 10750/10750 new unique smiles inputs (593033 persistent-cache hits; 635.2s)
Completed CHEMBL31: rows=1,541,941, primary=701,212, elapsed=914.5s


,release_label,schema_generation,rows,primary_record_eligible,relaxed_confidence_B_record_eligible,broad_prior_knowledge_eligible,pre15_multitarget_assays_excluded,post15_multicomponent_assays_excluded,unit_convertible,duplicate_flagged,requested_scientific_target_type,historical_native_target_type_equivalent
0,CHEMBL01,pre9_early,157058,108774,111963,117632,899,0,152223,0,SINGLE PROTEIN,PROTEIN
1,CHEMBL02,pre9_early,186089,61463,170030,179473,200,0,179801,0,SINGLE PROTEIN,PROTEIN
2,CHEMBL03,pre9_early,186184,60636,169320,180146,196,0,180474,0,SINGLE PROTEIN,PROTEIN
3,CHEMBL04,pre9_early,186185,60650,169334,180160,196,0,180488,0,SINGLE PROTEIN,PROTEIN
4,CHEMBL05,pre9_early,193304,61251,175160,186418,196,0,186749,0,SINGLE PROTEIN,PROTEIN
5,CHEMBL06,pre9_early,213950,61251,193705,206081,196,0,206428,0,SINGLE PROTEIN,PROTEIN
6,CHEMBL07,pre9_early,213894,61195,193649,206025,196,0,206372,0,SINGLE PROTEIN,PROTEIN
7,CHEMBL08,pre9_early,256300,93551,232773,246979,191,0,247350,0,SINGLE PROTEIN,PROTEIN
8,CHEMBL09,pre15_legacy,369031,156234,335594,353380,224,0,353980,0,SINGLE PROTEIN,PROTEIN
9,CHEMBL10,pre15_legacy,396500,169058,355233,379858,224,0,380523,0,SINGLE PROTEIN,PROTEIN


Historical extraction complete. Next: 04_F0_first_availability.ipynb
